<a href="https://colab.research.google.com/github/03sarath/gcp-ai-agents/blob/main/vertexai_ai_agent_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Copyright Psitron

# Intro to Building and Deploying an Agent with Agent Engine in Vertex AI

## Overview

### Gemini

Gemini is a family of generative AI models developed by Google DeepMind that is designed for multimodal use cases.

### Function Calling in Gemini

[Function calling](https://cloud.google.com/vertex-ai/docs/generative-ai/multimodal/function-calling) lets developers create a description of a function in their code, then pass that description to a language model in a request. The response from the model includes the name of a function that matches the description and the arguments to call it with.

### Agent Engine / Agent Runtime

[Agent Engine](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/runtime/quickstart) is a managed runtime that helps you build and deploy agents. It gives you the flexibility to choose how much reasoning you want to delegate to the LLM and how much you want to handle with customized code. You can define Python functions that get used as tools via Gemini Function Calling. Agent Engine works with LangChain, LangGraph, ADK, AG2, LlamaIndex, or custom Python code.

> **Naming note (2026):** Google rebranded Vertex AI's agent stack as the **Gemini Enterprise Agent Platform**, and **Agent Engine** is now also referred to as **Agent Runtime**. The API resource is still `reasoningEngines` and the Python module is still `vertexai.agent_engines`, so existing code keeps working - only the product names and docs URLs changed.

<img width="60%" src="https://storage.googleapis.com/github-repo/generative-ai/gemini/agent-engine/images/agent-engine-overview.png" alt="Agent Engine on Vertex AI" />

### Objectives

In this tutorial, you will learn how to build and deploy an agent (model, tools, and reasoning) using the Vertex AI SDK for Python.

You'll build and deploy an agent that uses the Gemini model, Python functions as tools, and LangChain for orchestration.

You will complete the following tasks:

- Install the Vertex AI SDK for Python
- Use the Vertex AI SDK to build components of a simple agent
- Test your agent locally before deploying
- Deploy and test your agent on Vertex AI
- Customize each layer of your agent (model, tools, orchestration)

### Costs

This tutorial uses billable components of Google Cloud:

- Vertex AI

Learn about [Vertex AI pricing](https://cloud.google.com/vertex-ai/pricing) and use the [Pricing Calculator](https://cloud.google.com/products/calculator/) to generate a cost estimate based on your projected usage.


## Getting Started


### Install Vertex AI SDK for Python

Install the Vertex AI SDK for Python with the extras needed for Agent Engine and LangChain.

The docs require **`google-cloud-aiplatform >= 1.112`**, which is the first version with the client-based `vertexai.Client()` / `client.agent_engines` API used below.

In [ ]:
%pip install --upgrade --quiet \
    "google-cloud-aiplatform[agent_engines,langchain]>=1.112" \
    requests

### Restart current runtime

To use the newly installed packages in this Jupyter runtime, you must restart the runtime. You can do this by running the cell below, which will restart the current kernel.

In [ ]:
# Restart kernel after installs so that your environment can access the new packages
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

<div class="alert alert-block alert-warning">
<b>⚠️ The kernel is going to restart. Please wait until it is finished before continuing to the next step. ⚠️</b>
</div>


### Authenticate your notebook environment (Colab only)

If you are running this notebook on Google Colab, run the following cell to authenticate your environment. This step is not required if you are using [Vertex AI Workbench](https://cloud.google.com/vertex-ai-workbench).

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Set Google Cloud project information and initialize the SDK

To get started you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

There are now **two** initialization styles, and this notebook uses both:

| Call | Used for |
|---|---|
| `vertexai.init(project=..., location=...)` | Building and **testing the agent locally** |
| `vertexai.Client(project=..., location=...)` | **Deploying and managing** the agent (`client.agent_engines.*`) |

Note that the staging bucket is no longer passed to `vertexai.init()` - it is now passed as `staging_bucket` inside the `config` of `client.agent_engines.create()`.

In [ ]:
PROJECT_ID = "testagnert"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}
STAGING_BUCKET = "gs://aiagentvertexaitst"  # @param {type:"string"}

import vertexai

# Used while you build and test the agent locally
vertexai.init(project=PROJECT_ID, location=LOCATION)

# Used to deploy and manage the agent on Agent Engine
client = vertexai.Client(project=PROJECT_ID, location=LOCATION)

## Example: Build and deploy an agent

### Import libraries

In [ ]:
from vertexai import agent_engines

### Define model

As you construct your agent from the bottom up, the first component deals with which generative model you want to use in your agent.

<img width="40%" src="https://storage.googleapis.com/github-repo/generative-ai/gemini/agent-engine/images/agent-stack-1.png" alt="Components of an agent in Agent Engine on Vertex AI" />

Here you'll use `gemini-2.5-flash`, which is generally available on regional endpoints such as `us-central1`.

Newer models are also available - `gemini-3.5-flash` (the one used in the current Agent Runtime LangChain quickstart), `gemini-3.8-flash`, and `gemini-3.1-pro-preview`. Some of the newest models are served **only on the global endpoint**, so if you switch to one and get a model-not-found error, set `LOCATION = "global"` in the setup cell above.

In [ ]:
model = "gemini-2.5-flash"

### Define Python functions (tools)

The second component of your agent includes tools and functions, which enable the generative model to interact with external systems, databases, document stores, and other APIs so that the model can get the most up-to-date information or take action with those systems.

<img width="40%" src="https://storage.googleapis.com/github-repo/generative-ai/gemini/agent-engine/images/agent-stack-2.png" alt="Components of an agent in Agent Engine on Vertex AI" />

In this example, you'll define a function called `get_exchange_rate` that uses the `requests` library to retrieve real-time currency exchange information from the [Frankfurter API](https://frankfurter.dev/).

> **The docstring is the tool schema.** Whatever you write in the docstring is what Gemini sees when it decides how to call the function - so any format constraint on an argument must be stated there. The `currency_date` argument is a good example: the API only accepts ISO `YYYY-MM-DD` or the literal `latest`. If the docstring doesn't say so, the model will happily pass a user's `07/09/2026` straight through and the call returns HTTP 404.
>
> Note also that the old `api.frankfurter.app` host now redirects to `api.frankfurter.dev/v1`, so we call the new host directly.

In [ ]:
def get_exchange_rate(
    currency_from: str = "USD",
    currency_to: str = "INR",
    currency_date: str = "latest",
):
    """Retrieves the exchange rate between two currencies on a specified date.

    Args:
        currency_from: 3-letter ISO 4217 code of the source currency, e.g. "USD".
        currency_to: 3-letter ISO 4217 code of the target currency, e.g. "INR".
        currency_date: The date in ISO 8601 format, YYYY-MM-DD (for example
            "2026-09-07"), or the literal string "latest" for the most recently
            published rate. No other date format is accepted - convert dates
            such as "07/09/2026" or "7 Sep 2026" to YYYY-MM-DD before calling.
            Rates are published on business days only, so a weekend or holiday
            date returns the previous business day's rate.

    Returns:
        A dict of exchange rate data, or a dict with an "error" key if the
        lookup failed.
    """
    import requests

    response = requests.get(
        f"https://api.frankfurter.dev/v1/{currency_date}",
        params={"from": currency_from, "to": currency_to},
        timeout=10,
    )

    if not response.ok:
        return {
            "error": (
                f"Exchange rate lookup failed with HTTP {response.status_code}. "
                f"currency_date must be 'latest' or an ISO date (YYYY-MM-DD); "
                f"got {currency_date!r}."
            )
        }

    return response.json()

Test the function with sample inputs to ensure that it's working as expected.

The second call deliberately uses a bad date format so that you can see the error message the agent would receive - that message tells the model exactly how to retry.

In [ ]:
print(get_exchange_rate(currency_from="USD", currency_to="INR"))
print(get_exchange_rate(currency_from="USD", currency_to="INR", currency_date="2026-09-04"))
print(get_exchange_rate(currency_from="USD", currency_to="INR", currency_date="07/09/2026"))

### Define agent

The third component of your agent involves adding a reasoning layer, which helps your agent use the tools that you provided to help the end user achieve a higher-level goal.

<img width="40%" src="https://storage.googleapis.com/github-repo/generative-ai/gemini/agent-engine/images/agent-stack-3.png" alt="Components of an agent in Agent Engine on Vertex AI" />

If you were to use Gemini and Function Calling on their own without a reasoning layer, you would need to handle the process of calling functions and APIs in your application code, and you would need to implement retries and additional logic to ensure that your function calling code is resilient to failures and malformed requests.

Here you'll use the LangChain agent template that ships with the SDK. Note the import path: it is now **`vertexai.agent_engines.LangchainAgent`**. The old `vertexai.preview.reasoning_engines.LangchainAgent` path is the legacy preview namespace and should no longer be used.

In [ ]:
agent = agent_engines.LangchainAgent(
    model=model,
    tools=[get_exchange_rate],
    model_kwargs={
        "temperature": 0.28,
        "max_output_tokens": 1000,
        "top_p": 0.95,
    },
    agent_executor_kwargs={"return_intermediate_steps": True},
)

Now we can test the model and agent behavior to ensure that it's working as expected before we deploy it:

### Test your agent locally

With all of the core components of your agent in place, you can send a prompt to your agent using `.query` to test that it's working as expected, including the intermediate steps that the agent performed between the input prompt and the generated summary output. In the default mode, the agent processes your input and returns the **entire agent output in a single response when complete**:

In [ ]:
agent.query(
    input="What's the exchange rate from US dollars to INR currency on 2026-09-04?"
)

In addition to the default query mode, the `.stream_query` method allows you to **see the agent's intermediate steps and final output from the chain**.

Instead of waiting for the agent to complete all sub-tasks, the agent sends back the response in **chunks as it's being generated**:

In [ ]:
message_types = {"actions": "Action", "messages": "Message", "output": "Output"}
for chunk in agent.stream_query(
    input="What's the exchange rate from US dollars to INR currency today?"
):
    for key, label in message_types.items():
        if key in chunk:
            print("\n------\n")
            print(f"{label}:")
            print()
            print(chunk[key])

This allows you to observe the agent's actions in real-time (such as function calls, and intermediate steps), which is helpful for debugging purposes or for providing real-time updates to the end user.

### Deploy your agent on Vertex AI

Now that you've specified a model, tools, and reasoning for your agent and tested it out, you're ready to deploy your agent as a remote service in Vertex AI!

<img width="40%" src="https://storage.googleapis.com/github-repo/generative-ai/gemini/agent-engine/images/agent-stack-4.png" alt="Components of an agent in Agent Engine on Vertex AI" />

You can re-define the agent to avoid any stateful information in the agent due to our testing in the previous cell:

In [ ]:
agent = agent_engines.LangchainAgent(
    model=model,
    tools=[get_exchange_rate],
    model_kwargs={
        "temperature": 0.28,
        "max_output_tokens": 1000,
        "top_p": 0.95,
    },
)

Now you're ready to deploy your agent by calling **`client.agent_engines.create()`** with:

1. The instance of your agent (`agent=...`)
2. A `config` dict holding the deployment settings - `display_name`, the `requirements` your agent needs at runtime (like a `requirements.txt`), and the `staging_bucket` used to upload your pickled agent.

> **What changed:** the old top-level `agent_engines.create(agent, requirements=[...])` helper has been replaced by the client-based `client.agent_engines.create(agent=..., config={...})`, and `staging_bucket` moved from `vertexai.init()` into `config`.

In [ ]:
remote_agent = client.agent_engines.create(
    agent=agent,
    config={
        "display_name": "LangChain currency exchange agent",
        "requirements": [
            "google-cloud-aiplatform[agent_engines,langchain]>=1.112",
            "requests",
        ],
        "staging_bucket": STAGING_BUCKET,
    },
)

Now you can send a prompt to your remote agent using `.query` to test that it's working as expected:

In [ ]:
remote_agent.query(
    input="What's the exchange rate from US dollars to INR currency on 2026-09-04?"
)

Or you can stream the results back from the remote agent interactively using `.stream_query`:

In [ ]:
message_types = {"actions": "Action", "messages": "Message", "output": "Output"}
for chunk in remote_agent.stream_query(
    input="What's the exchange rate from US dollars to SEK currency today?"
):
    for key, label in message_types.items():
        if key in chunk:
            print("\n------\n")
            print(f"{label}:")
            print()
            print(chunk[key])

### Querying your deployed agent

You've now deployed your agent and can interact with it in multiple ways, both within this notebook and from other applications or environments. The primary methods for accessing your deployed agent are via the Python client library or through REST API calls. Here's an overview of both methods:

**Method 1: Reusing within this notebook or another Python environment**

You can directly reuse and query the `remote_agent` instance you created in this notebook.

Or, you can instantiate a new instance in another notebook or Python script. To do this, you'll need your deployed agent's resource name - a string that includes the project, location, and Agent Engine ID. With the client-based SDK you read it from **`remote_agent.api_resource.name`** (the old `remote_agent.resource_name` attribute belonged to the legacy API):

In [ ]:
remote_agent.api_resource.name

Use the resource name to load the agent in your other notebook or Python script with **`client.agent_engines.get()`**, then query the remote agent as usual:

In [ ]:
import vertexai

AGENT_ENGINE_RESOURCE_NAME = "projects/1006336035909/locations/us-central1/reasoningEngines/8155162955365744640"  # Replace with the resource name of your deployed agent

client = vertexai.Client(project=PROJECT_ID, location=LOCATION)

remote_agent = client.agent_engines.get(name=AGENT_ENGINE_RESOURCE_NAME)
response = remote_agent.query(
    input="What's the exchange rate from US dollars to SEK currency today?"
)
print(response)

**Method 2: Accessing from other environments via REST API**

Beyond the Python client library, your deployed agent can be queried using REST API calls:

- Python: You can use Python's `requests` library or similar tools to make HTTP calls to the REST API.
- cURL: A command-line tool, cURL allows you to send HTTP requests directly. This is useful for testing and debugging.
- Other Programming Languages: If you prefer a different language for your application, you can use its native HTTP client library to make REST API calls.

The request body takes two fields: `class_method` (which method on your agent to invoke - optional, defaults to `query`) and `input` (the arguments passed to that method). For a `LangchainAgent`, `query` takes a single `input` string, which is why `input` is nested.

Template:

```bash
curl \
  -H "Authorization: Bearer $(gcloud auth print-access-token)" \
  -H "Content-Type: application/json" \
  "https://LOCATION-aiplatform.googleapis.com/v1/projects/PROJECT_ID/locations/LOCATION/reasoningEngines/RESOURCE_ID:query" \
  -d '{
    "class_method": "query",
    "input": {
      "input": "What is the exchange rate from US dollars to Swedish currency?"
    }
  }'
```

Filled-in sample:

```bash
curl \
  -H "Authorization: Bearer $(gcloud auth print-access-token)" \
  -H "Content-Type: application/json" \
  "https://us-central1-aiplatform.googleapis.com/v1/projects/aiagentvertexai/locations/us-central1/reasoningEngines/8189839902838358016:query" \
  -d '{
    "class_method": "query",
    "input": {
      "input": "What is the exchange rate from US dollars to INR currency?"
    }
  }'
```

To stream the response instead, POST the same body to the `:streamQuery` endpoint with `"class_method": "stream_query"`.

Post Man:
## Postman Setup Instructions

---

### 1. **Method & URL**

- **Method**: `POST`
- **URL**:
`https://us-central1-aiplatform.googleapis.com/v1/projects/aiagentvertexai/locations/us-central1/reasoningEngines/8189839902838358016:query`

---

### 2. **Authorization**

- **Type**: Bearer Token

#### How to Get Token:

Run the following command in your terminal:

```bash
gcloud auth print-access-token
```

In Postman:

Go to the Authorization tab

Set Type to Bearer Token

Paste the token in the Token field

### 3. Headers
Add the following headers if not using the Authorization tab:

| Key | Value |
|---|---|
| Authorization | Bearer YOUR_ACCESS_TOKEN |
| Content-Type | application/json |

### 4. Body
Go to the Body tab
Select raw
Choose JSON from the format dropdown
Paste the following JSON payload:

```json
{
  "class_method": "query",
  "input": {
    "input": "What is the exchange rate from US dollars to INR currency?"
  }
}
```

## Cleaning up

After you've finished, it's a good practice to clean up your cloud resources. Delete the deployed agent to avoid any unexpected charges on your Google Cloud account.

`force=True` also deletes any child resources (such as sessions and memories) that the agent created.

In [ ]:
remote_agent.delete(force=True)